# 02 — Nettoyage

**Cartographier les manques des ERP** · Ameth Yade · M2 SIGLIS, Université de Pau et des Pays de l'Adour

---

Le nettoyage est la majeure partie du travail réel d'un analyste, et il est bon que
cela se voie. **Chaque décision prise ici est écrite et justifiée avant d'être
appliquée** : c'est ce qui distingue un travail sérieux d'un script jetable.

Aucune de ces décisions n'est neutre. Une autre personne trancherait autrement, et les
résultats bougeraient. Les exposer, c'est permettre qu'on les conteste.

In [ ]:
import re
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore", message=".*OpenSSL.*")
pd.set_option("display.width", 120)

brut = pd.read_csv("../data/raw/modules_raw.csv")
print(f"{len(brut):,} lignes, {len(brut.columns)} colonnes")
brut.head(3)

: 

## Décision 1 — L'unité d'observation

**Une ligne = un couple (module, série Odoo).** Le même module publié pour six versions
donne six lignes.

Ce n'est pas une fatalité, c'est un choix. L'alternative — une ligne par module —
répondrait à « combien de modules distincts existe-t-il ». Mais la question du projet
porte sur **l'offre disponible pour une version donnée**, et sur son évolution : c'est
le couple qui la porte.

Conséquence à ne jamais oublier dans les analyses : *compter des lignes, ce n'est pas
compter des modules*. Pour les modules distincts, on déduplique sur `technical_name`.

In [ ]:
print(f"Lignes (module x serie) : {len(brut):,}")
print(f"Modules distincts       : {brut.technical_name.nunique():,}")
print(f"Ratio                   : {len(brut) / brut.technical_name.nunique():.2f} series par module")

## Décision 2 — Les doublons, relevés avant d'être retirés

Un même couple (nom technique, série) peut apparaître deux fois : le module est alors
rangé dans deux catégories.

**On ne supprime pas à l'aveugle.** Le fait qu'un module relève de plusieurs rubriques
est une information sur la taxonomie, pas une erreur de collecte. On la relève d'abord,
on déduplique ensuite.

In [ ]:
df = brut.copy()
cle = ["technical_name", "series"]

# Releve AVANT deduplication : apres, l'information serait perdue.
df["n_categories"] = df.groupby(cle)["source_category"].transform("nunique")
df["categories_all"] = df.groupby(cle)["source_category"].transform(
    lambda s: " | ".join(sorted(set(s))))

avant = len(df)
multi = (df["n_categories"] > 1).mean()
df = df.drop_duplicates(subset=cle, keep="first").reset_index(drop=True)

print(f"Doublons retires  : {avant - len(df):,} sur {avant:,} ({(avant - len(df)) / avant * 100:.1f} %)")
print(f"Modules ranges dans plus d'une categorie : {multi * 100:.1f} %")

## Décision 3 — Normaliser les éditeurs

« Cybrosys Techno Solutions » et « Cybrosys Techno Solutions Pvt. Ltd. » sont le même
éditeur. Sans normalisation, toute mesure de concentration du marché est fausse.

Un détail d'ordre compte : **les suffixes juridiques doivent être retirés avant la
ponctuation**. Si l'on efface les points d'abord, `s.a.` devient `s a` et échappe au
filtre. C'est le genre de bug qui ne se voit pas dans les totaux.

L'heuristique a un coût : elle peut fusionner deux éditeurs aux noms proches. On
conserve donc toujours le nom d'origine à côté du nom normalisé.

In [ ]:
SUFFIXES = re.compile(
    r"\b(pvt\.?|private|ltd\.?|limited|llp|llc|inc\.?|incorporated|gmbh|"
    r"s\.?a\.?r\.?l\.?|sarl|sas|s\.?a\.?|bv|b\.?v\.?|nv|oy|ab|as|aps|srl|"
    r"s\.?r\.?l\.?|co\.?|corp\.?|corporation|company|technologies|technology|"
    r"solutions|software|systems|it|group|holding)\b", re.I)


def normaliser_editeur(nom):
    if not isinstance(nom, str):
        return ""
    t = nom.lower()
    t = SUFFIXES.sub(" ", t)          # avant la ponctuation : << s.a. >> reste lisible
    t = re.sub(r"[.,&()/|]+", " ", t)
    t = SUFFIXES.sub(" ", t)          # apres : attrape << sarl >> ecrit sans points
    t = re.sub(r"[^a-z0-9\s-]", " ", t)
    t = " ".join(x for x in t.split() if len(x) > 1)  # residus de sigles
    return re.sub(r"\s+", " ", t).strip()


for n in ["Cybrosys Techno Solutions", "Cybrosys Techno Solutions Pvt. Ltd.",
          "Odoo S.A.", "Odoo SA", "Odoo", "Odoo Mates", "Kanak Infosystems LLP."]:
    print(f"  {n:<38} -> {normaliser_editeur(n)!r}")

In [ ]:
df["author_clean"] = df["author"].map(normaliser_editeur)
# Un nom vide apres normalisation = un editeur dont le nom n'etait que des
# suffixes. On retombe sur le nom brut plutot que de perdre la ligne.
vide = df["author_clean"] == ""
df.loc[vide, "author_clean"] = df.loc[vide, "author"].fillna("").str.lower()

print(f"Editeurs (bruts)      : {df.author.nunique():,}")
print(f"Editeurs (normalises) : {df.author_clean.nunique():,}")
print(f"Regroupements operes  : {df.author.nunique() - df.author_clean.nunique():,}")

## Décision 4 — Prix et compteurs d'usage

**Un module gratuit a un prix de 0, pas un prix manquant.** La distinction compte : si
l'on laisse `NaN`, toute moyenne de prix exclut silencieusement les gratuits et
surestime le marché.

Un module payant dont le montant n'a pas pu être lu garde `NaN` : on ne devine pas.

Pour les compteurs, le piège est plus sournois. Les modules gratuits affichent des
**téléchargements**, les payants des **achats**. Ce ne sont pas la même grandeur.
Les réunir dans une colonne « usage » produirait un chiffre qui ne veut rien dire.

In [ ]:
for col in ("price_value", "n_downloads", "n_purchases", "n_last_month"):
    df[col] = pd.to_numeric(df[col], errors="coerce")
df["is_free"] = df["is_free"].astype(str).str.lower().isin(["true", "1"])

df.loc[df["is_free"], "price_value"] = 0.0
df["price_missing"] = (~df["is_free"]) & df["price_value"].isna()

df["usage_kind"] = "none"
df.loc[df.n_downloads.notna(), "usage_kind"] = "downloads"
df.loc[df.n_purchases.notna(), "usage_kind"] = "purchases"

print(f"Gratuits                : {df.is_free.mean() * 100:.1f} %")
print(f"Prix manquant (payants) : {df.price_missing.sum():,}")
print(f"\nType de compteur :")
print(df.usage_kind.value_counts().to_string())
print(f"\nDevises :")
print(df.price_currency.value_counts(dropna=False).head().to_string())

## Décision 5 — Ordonner les séries

`"10.0" < "9.0"` en tri alphabétique. Toute courbe temporelle construite là-dessus
serait fausse — et le graphique aurait l'air normal.

Les releases intermédiaires d'Odoo Online (`saas-19.4`) se placent juste après la
version majeure correspondante.

In [ ]:
def ordre_serie(s):
    if not isinstance(s, str):
        return -1.0
    m = re.search(r"(\d+)\.(\d+)", s)
    if not m:
        return -1.0
    return int(m.group(1)) + int(m.group(2)) / 100 + (0.001 if s.startswith("saas") else 0.0)


df["series_order"] = df["series"].map(ordre_serie)
df["is_saas"] = df["series_kind"].eq("saas")

apercu = df[["series", "series_order"]].drop_duplicates().sort_values("series_order")
print(f"{len(apercu)} series, dans l'ordre :")
print("  " + "  ".join(apercu.series.tolist()))
print(f"\nLignes en release intermediaire (saas-*) : {df.is_saas.sum():,}")

## Décision 6 — Séparer la prose des mots-clés

Certains résumés sont gonflés de mots-clés de référencement, introduits par
`Search Keywords:`. Sur un cas rencontré, le texte passait de 1 240 à 174 caractères
une fois les mots-clés retirés.

**Mesuré sur le corpus complet, le phénomène est marginal : 49 résumés sur 72 316,
soit 0,05 %.** Le cas qui avait motivé ce traitement était un cas isolé, et il aurait
été facile d'en tirer une règle générale sans vérifier.

On conserve néanmoins la séparation. Elle ne coûte rien, elle protège l'analyse
lexicale d'un biais réel bien que rare, et `has_seo_keywords` documente son ampleur
plutôt que de la laisser supposer.

La vraie difficulté des résumés est ailleurs : ils sont **très inégaux**. Médiane de
71 caractères, moyenne de 273, maximum à 77 007 — et près d'un sur dix est vide.

In [ ]:
df["summary"] = df["summary"].fillna("").str.strip()
df["summary_missing"] = df["summary"].str.len().eq(0)

decoupe = df["summary"].str.split(r"(?i)\|?\s*search keywords\s*:", n=1, regex=True)
df["summary_prose"] = decoupe.str[0].str.strip()
df["has_seo_keywords"] = decoupe.str.len().gt(1)

print(f"Resumes vides              : {df.summary_missing.sum():,} ({df.summary_missing.mean() * 100:.1f} %)")
print(f"Resumes avec mots-cles SEO : {df.has_seo_keywords.mean() * 100:.1f} %")
print(f"Longueur mediane, brute    : {df.summary.str.len().median():.0f} caracteres")
print(f"Longueur mediane, prose    : {df.summary_prose.str.len().median():.0f} caracteres")

## Décision 7 — Un référentiel fonctionnel unique

La taxonomie du site n'a pas été conçue pour l'analyse. Elle contient :

- des **quasi-doublons** : `Purchase` / `Purchases`, `Inventory` / `Warehouse`,
  `Localization` / `Localisation`, `Other` / `Others` ;
- des **rubriques fourre-tout** : `Tools` et `Extra Tools`, 24 205 modules à elles deux ;
- des **catégories de trois modules**.

Compter par catégorie brute reviendrait à mesurer les habitudes de rangement des
éditeurs plutôt que la structure du besoin.

La table de correspondance ci-dessous est une **interprétation**, pas une vérité. Elle
est écrite dans un fichier versionné (`data/processed/category_mapping.csv`) pour
pouvoir être relue, contestée et modifiée sans toucher au code.

Deux choix méritent d'être signalés dans le document final :

- **`Tools` et `Extra Tools` ne sont pas réparties de force.** Leur poids — 29 % du
  catalogue — est un résultat en soi : le besoin le plus massif est celui que la
  taxonomie ne sait pas nommer. Les diluer effacerait ce constat.
- **`Uncategorized`, `Other`, `Others`, `Hidden` sont conservées**, groupées en
  « Non renseigné ». Les supprimer gonflerait artificiellement les autres fonctions.

In [ ]:
GROUPES = {
    "Comptabilite & finance": ["Accounting", "Finance", "Invoicing", "Payment",
                               "Payment Providers", "Expenses"],
    "Ventes & CRM": ["Sales", "CRM", "Subscription", "Subscriptions", "Rental",
                     "Appointment"],
    "Achats": ["Purchase", "Purchases"],
    "Point de vente": ["Point Of Sale", "Restaurant"],
    "Logistique & stocks": ["Inventory", "Warehouse", "Delivery", "Shipping",
                            "Logistics", "Transport", "Barcode"],
    "Production": ["Manufacturing", "PLM", "Quality", "Maintenance", "Repair"],
    "Ressources humaines": ["Human Resources", "Payroll", "Time Off", "Recruitment",
                            "Timesheets", "Appraisals", "Lunch", "Fleet"],
    "Site web & e-commerce": ["Website", "Ecommerce", "Blog", "Forum", "Theme",
                              "Website Theme", "Frontend", "Portal"],
    "Marketing": ["Marketing", "Social Marketing", "Email Marketing", "SMS Marketing",
                  "Marketing Automation", "Mass Mailing", "Events", "Surveys"],
    "Projets & services": ["Project", "Services", "Field Service", "Helpdesk", "Planning"],
    "Collaboration & documents": ["Productivity", "Discuss", "Documents",
                                  "Document Management", "Knowledge", "Sign", "Notes",
                                  "Approvals", "WhatsApp", "VoIP", "Live Chat", "eLearning"],
    "Decisionnel & reporting": ["Reporting", "Dashboards", "Spreadsheet", "Data Cleaning"],
    "Localisation & conformite": ["Localization", "Localisation"],
    "Verticaux sectoriels": ["Industries", "Education", "Real Estate", "Agriculture",
                             "Construction", "Healthcare", "Hotel", "Government", "Nonprofit"],
    "Integrations & connecteurs": ["Connector", "Integration", "IoT",
                                   "Internet of Things", "Artificial Intelligence", "API"],
    "Outils transverses & technique": ["Tools", "Extra Tools", "Technical",
                                       "Administration", "Generic Modules",
                                       "Customizations", "Security", "Usability",
                                       "Extra Rights", "Backend", "Utilities",
                                       "Migration", "Backup", "Mobile", "Widgets", "Studio"],
    "Non renseigne": ["Uncategorized", "Other", "Others", "Hidden", "Tutorial"],
}

correspondance = {c: f for f, membres in GROUPES.items() for c in membres}

# Deux controles bloquants. Une categorie oubliee disparaitrait sinon des
# totaux sans que rien ne le signale -- exactement le type d'erreur que le
# notebook precedent a mis au jour.
cats_site = set(pd.read_csv("../data/processed/categories.csv").category)
manquantes = sorted(cats_site - set(correspondance))
inconnues = sorted(set(correspondance) - cats_site)
assert not manquantes, f"categories du site non affectees : {manquantes}"
assert not inconnues, f"categories inconnues du site : {inconnues}"

print(f"{len(cats_site)} categories -> {len(GROUPES)} fonctions metier. Aucune perte.")

In [ ]:
df["fonction"] = df["source_category"].map(correspondance)
non_mappees = df.fonction.isna().sum()
print(f"Lignes sans fonction : {non_mappees:,}")

(pd.DataFrame(sorted(correspondance.items()), columns=["category", "fonction"])
   .to_csv("../data/processed/category_mapping.csv", index=False))

repartition = df.fonction.value_counts()
largeur = max(len(f) for f in repartition.index)
for fonction, n in repartition.items():
    barre = "#" * round(n / repartition.max() * 45)
    print(f"  {fonction:<{largeur}} {n:>7,}  {n / len(df) * 100:5.1f} %  {barre}")

## Le dictionnaire des données

Un jeu de données sans dictionnaire n'est pas réutilisable. Celui-ci est écrit dans
`data/processed/dictionnaire_donnees.md`.

In [ ]:
DICTIONNAIRE = [
    ("technical_name", "texte", "Nom technique du module, unique par serie"),
    ("series", "texte", "Version Odoo visee (5.0 a 19.0, ou saas-XX.Y)"),
    ("series_kind", "texte", "major = version majeure ; saas = release intermediaire Odoo Online"),
    ("series_order", "decimal", "Cle de tri numerique des series (evite 10.0 < 9.0)"),
    ("is_saas", "booleen", "Vrai pour les releases intermediaires"),
    ("title", "texte", "Titre commercial affiche"),
    ("author", "texte", "Nom de l'editeur, tel qu'affiche"),
    ("author_clean", "texte", "Nom normalise : minuscules, suffixes juridiques retires"),
    ("is_free", "booleen", "Vrai si le module est gratuit"),
    ("price_value", "decimal", "Prix ; 0 pour les gratuits ; vide si illisible"),
    ("price_currency", "texte", "Devise affichee ; vide pour les gratuits"),
    ("price_missing", "booleen", "Vrai si module payant dont le prix n'a pas pu etre lu"),
    ("n_downloads", "entier", "Telechargements cumules (modules GRATUITS uniquement)"),
    ("n_purchases", "entier", "Achats cumules (modules PAYANTS uniquement)"),
    ("n_last_month", "entier", "Telechargements ou achats du dernier mois"),
    ("usage_kind", "texte", "downloads | purchases | none : quelle grandeur est mesuree"),
    ("summary", "texte", "Resume affiche sur la carte, mots-cles SEO compris"),
    ("summary_prose", "texte", "Resume ampute de la section << Search Keywords >>"),
    ("has_seo_keywords", "booleen", "Vrai si le resume contenait des mots-cles SEO"),
    ("summary_missing", "booleen", "Vrai si le resume est vide"),
    ("image_id", "texte", "Identifiant de la vignette sur le CDN Odoo"),
    ("source_category", "texte", "Categorie du site d'ou la fiche a ete lue"),
    ("fonction", "texte", "Fonction metier du referentiel (17 valeurs)"),
    ("n_categories", "entier", "Nombre de categories du site ou le module apparait"),
    ("categories_all", "texte", "Liste de ces categories, separees par |"),
    ("source_url", "texte", "URL de la page de liste d'origine"),
]

lignes = [
    "# Dictionnaire des donnees",
    "",
    "Fichier : `data/processed/modules_clean.csv`",
    "",
    "**Unite d'observation : un couple (module, serie Odoo).** Un module publie pour",
    "six versions occupe six lignes. Compter des lignes n'est donc pas compter des",
    "modules : pour les modules distincts, dedupliquer sur `technical_name`.",
    "",
    "| Colonne | Type | Description |",
    "|---|---|---|",
]
lignes += [f"| `{c}` | {t} | {d} |" for c, t, d in DICTIONNAIRE]
lignes += [
    "",
    "## Avertissements",
    "",
    "- `n_downloads` et `n_purchases` **ne mesurent pas la meme chose** et ne doivent",
    "  jamais etre additionnes. `usage_kind` indique laquelle est renseignee.",
    "- `fonction` resulte d'une table de correspondance interpretative",
    "  (`category_mapping.csv`), pas d'une donnee du site.",
    "- La note moyenne, la licence et la date de publication ne figurent pas sur les",
    "  cartes de liste et sont donc absentes du corpus.",
]

Path("../data/processed/dictionnaire_donnees.md").write_text("\n".join(lignes), encoding="utf-8")
print(f"{len(DICTIONNAIRE)} colonnes documentees")

## Bilan et export

In [ ]:
print(f"Lignes (module x serie)   : {len(df):,}")
print(f"Modules distincts         : {df.technical_name.nunique():,}")
print(f"Editeurs normalises       : {df.author_clean.nunique():,}")
print(f"Series                    : {df.series.nunique()}")
print(f"Fonctions metier          : {df.fonction.nunique()}")
print(f"Gratuits                  : {df.is_free.mean() * 100:.1f} %")
print(f"Resumes vides             : {df.summary_missing.mean() * 100:.1f} %")

df.to_csv("../data/processed/modules_clean.csv", index=False)
print(f"\nEcrit : data/processed/modules_clean.csv")

---

**Sorties de ce notebook**

| Fichier | Contenu |
|---|---|
| `data/processed/modules_clean.csv` | le corpus nettoyé |
| `data/processed/category_mapping.csv` | la table de correspondance |
| `data/processed/dictionnaire_donnees.md` | le dictionnaire des données |

Suite : **`03_analyse.ipynb`**.